<a href="https://colab.research.google.com/github/LatiefDataVisionary/pm-tools-absa-analysis/blob/main/notebook/KDK_Scraping_Ulasan_dari_X.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install playwright
!playwright install chromium
!playwright install-deps chromium

(node:3628) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
167.3 MiB [] 0% 0.0s167.3 MiB [] 0% 21.0s167.3 MiB [] 0% 11.6s167.3 MiB [] 0% 9.0s167.3 MiB [] 1% 3.6s167.3 MiB [] 3% 2.6s167.3 MiB [] 3% 2.4s167.3 MiB [] 5% 2.1s167.3 MiB [] 6% 1.9s167.3 MiB [] 7% 1.7s167.3 MiB [] 9% 1.6s167.3 MiB [] 10% 1.5s167.3 MiB [] 12% 1.4s167.3 MiB [] 13% 1.3s167.3 MiB [] 14% 1.3s167.3 MiB [] 15% 1.3s167.3 MiB [] 17% 1.2s167.3 MiB [] 18% 1.2s167.3 MiB [] 19% 1.2s167.3 MiB [] 21% 1.1s167.3 MiB [] 23% 1.1s167.3 MiB [] 25% 1.0s167.3 MiB [] 26% 1.0s167.3 MiB [] 27% 1.0s167.3 MiB [] 28% 1.0s167.3 MiB [] 30% 0.9s167.3 MiB [] 31% 0.9s167.3 MiB [] 33% 0.9s167.3 MiB [] 35% 0.8s167.3 MiB [] 37% 0.8s167.3 MiB [] 39% 0.7s167.3 MiB [] 40% 0.7s167.3 MiB [] 41% 0.7s167.3 MiB [] 43%

In [17]:
import asyncio
import json
import pandas as pd
from pathlib import Path
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

In [18]:
class XDataExtractor:
    def __init__(self, cookies_path: str, max_scrolls: int = 5):
        self.cookies_path = Path(cookies_path)
        self.max_scrolls = max_scrolls
        self.user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"

    async def _load_cookies(self, context) -> bool:
        if not self.cookies_path.exists():
            print(f"CRITICAL: File {self.cookies_path} tidak ditemukan.")
            return False

        with open(self.cookies_path, 'r') as f:
            cookies = json.load(f)

        sanitized_cookies = []
        for cookie in cookies:
            # Mapping nilai non-standar ke standar Playwright
            if "sameSite" in cookie:
                ss = str(cookie["sameSite"]).lower()
                if ss in ["unspecified", "no_restriction"]:
                    cookie["sameSite"] = "Lax"
                else:
                    cookie["sameSite"] = ss.capitalize()

            # Hapus atribut internal browser yang sering bikin error
            cookie.pop("id", None)
            cookie.pop("storeId", None)
            sanitized_cookies.append(cookie)

        await context.add_cookies(sanitized_cookies)
        return True

    async def _parse_tweets(self, page) -> list:
        tweets_data = []
        # Target data-testid yang lebih stabil
        articles = await page.query_selector_all('article[data-testid="tweet"]')

        for article in articles:
            try:
                text_node = await article.query_selector('[data-testid="tweetText"]')
                time_node = await article.query_selector('time')

                if text_node:
                    content = await text_node.inner_text()
                    tweets_data.append({
                        "timestamp": await time_node.get_attribute('datetime') if time_node else None,
                        "text": content.replace('\n', ' ').strip()
                    })
            except:
                continue
        return tweets_data

    async def scrape_keyword(self, keyword: str) -> pd.DataFrame:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            context = await browser.new_context(user_agent=self.user_agent)

            if not await self._load_cookies(context):
                await browser.close()
                return pd.DataFrame()

            page = await context.new_page()
            # Live search agar dapat post terbaru
            search_url = f"https://x.com/search?q={keyword}&f=live"

            all_tweets = []
            seen_texts = set()

            try:
                print(f"Navigasi ke: {search_url}")
                await page.goto(search_url, wait_until="domcontentloaded", timeout=60000)

                # Cek apakah kita kena login wall atau berhasil masuk
                try:
                    await page.wait_for_selector('[data-testid="SideNav_NewTweet_Button"]', timeout=15000)
                    print("Login Berhasil (Session Valid)")
                except:
                    print("PERINGATAN: Tidak mendeteksi tombol Tweet. Mungkin cookie tidak valid/expired.")

                for scroll in range(self.max_scrolls):
                    await asyncio.sleep(2)
                    current_batch = await self._parse_tweets(page)

                    for t in current_batch:
                        if t['text'] not in seen_texts:
                            seen_texts.add(t['text'])
                            t['target_keyword'] = keyword
                            all_tweets.append(t)

                    await page.evaluate("window.scrollBy(0, 2000)")

            except Exception as e:
                print(f"Error saat scraping {keyword}: {str(e)}")
            finally:
                await browser.close()

            return pd.DataFrame(all_tweets)

In [21]:
async def main():
    # Meningkatkan max_scrolls ke 50 untuk mendapatkan ratusan data jika tersedia
    extractor = XDataExtractor(cookies_path="/content/cookies.json", max_scrolls=50)

    # Menggunakan kombinasi query OR untuk cakupan maksimal
    targets = {
        "Trello": "@trello OR trello OR #trello",
        "Jira": "@jira OR jira OR #jira",
        "ClickUp": "@clickup OR clickup OR #clickup"
    }

    for name, query in targets.items():
        print(f"--- Memulai Misi Scraping: {name} ---")
        print(f"Query: {query}")

        df = await extractor.scrape_keyword(query)

        if not df.empty:
            filename = f"{name.lower()}_x_reviews.csv"
            df.to_csv(filename, index=False)
            print(f"BERHASIL: {len(df)} data {name} diamankan di {filename}.")
        else:
            print(f"PERINGATAN: Tidak ada data ditemukan untuk {name}. Cek validitas cookie.")

    print("\n--- SEMUA MISI SELESAI DENGAN TOTALITAS --- ")

# Jalankan eksekusi
await main()

--- Memulai Misi Scraping: Trello ---
Query: @trello OR trello OR #trello
Navigasi ke: https://x.com/search?q=@trello OR trello OR #trello&f=live
Login Berhasil (Session Valid)
BERHASIL: 179 data Trello diamankan di trello_x_reviews.csv.
--- Memulai Misi Scraping: Jira ---
Query: @jira OR jira OR #jira
Navigasi ke: https://x.com/search?q=@jira OR jira OR #jira&f=live
Login Berhasil (Session Valid)
BERHASIL: 139 data Jira diamankan di jira_x_reviews.csv.
--- Memulai Misi Scraping: ClickUp ---
Query: @clickup OR clickup OR #clickup
Navigasi ke: https://x.com/search?q=@clickup OR clickup OR #clickup&f=live
Login Berhasil (Session Valid)
PERINGATAN: Tidak ada data ditemukan untuk ClickUp. Cek validitas cookie.

--- SEMUA MISI SELESAI DENGAN TOTALITAS --- 


In [20]:
# Menjalankan fungsi utama untuk memulai scraping
# Pastikan file /content/cookies.json sudah ada sebelum menjalankan ini
await main()

Memulai scraping untuk: @trello
Navigasi ke: https://x.com/search?q=@trello&f=live
Login Berhasil (Session Valid)
Selesai scraping @trello. Data disimpan di trello_x_reviews.csv. Total data: 46
Memulai scraping untuk: @jira
Navigasi ke: https://x.com/search?q=@jira&f=live
Login Berhasil (Session Valid)
Selesai scraping @jira. Data disimpan di jira_x_reviews.csv. Total data: 76
Memulai scraping untuk: @clickup
Navigasi ke: https://x.com/search?q=@clickup&f=live
Login Berhasil (Session Valid)
Selesai scraping @clickup. Data disimpan di clickup_x_reviews.csv. Total data: 51
Semua scraping selesai!


In [22]:
async def main():
    # Meningkatkan max_scrolls ke 50 untuk mendapatkan ratusan data jika tersedia
    extractor = XDataExtractor(cookies_path="/content/cookies.json", max_scrolls=50)

    # Menggunakan kombinasi query OR untuk cakupan maksimal
    targets = {
        "Trello": "@trello OR trello OR #trello",
        "Jira": "@jira OR jira OR #jira",
        "ClickUp": "@clickup OR clickup OR #clickup"
    }

    for name, query in targets.items():
        print(f"--- Memulai Misi Scraping: {name} ---")
        print(f"Query: {query}")

        df = await extractor.scrape_keyword(query)

        if not df.empty:
            filename = f"{name.lower()}_x_reviews_2.csv"
            df.to_csv(filename, index=False)
            print(f"BERHASIL: {len(df)} data {name} diamankan di {filename}.")
        else:
            print(f"PERINGATAN: Tidak ada data ditemukan untuk {name}. Cek validitas cookie.")

    print("\n--- SEMUA MISI SELESAI DENGAN TOTALITAS --- ")

# Jalankan eksekusi
await main()

--- Memulai Misi Scraping: Trello ---
Query: @trello OR trello OR #trello
Navigasi ke: https://x.com/search?q=@trello OR trello OR #trello&f=live
Login Berhasil (Session Valid)
BERHASIL: 176 data Trello diamankan di trello_x_reviews_2.csv.
--- Memulai Misi Scraping: Jira ---
Query: @jira OR jira OR #jira
Navigasi ke: https://x.com/search?q=@jira OR jira OR #jira&f=live
Login Berhasil (Session Valid)
BERHASIL: 119 data Jira diamankan di jira_x_reviews_2.csv.
--- Memulai Misi Scraping: ClickUp ---
Query: @clickup OR clickup OR #clickup
Navigasi ke: https://x.com/search?q=@clickup OR clickup OR #clickup&f=live
Login Berhasil (Session Valid)
BERHASIL: 131 data ClickUp diamankan di clickup_x_reviews_2.csv.

--- SEMUA MISI SELESAI DENGAN TOTALITAS --- 


In [23]:
import pandas as pd
import glob

def merge_and_deduplicate_csv(keyword_prefix):
    all_files = glob.glob(f"/content/{keyword_prefix}_x_reviews*.csv")
    if not all_files:
        print(f"Tidak ada file CSV ditemukan untuk prefix: {keyword_prefix}_x_reviews")
        return pd.DataFrame()

    li = []
    for filename in all_files:
        df = pd.read_csv(filename, index_col=None, header=0)
        li.append(df)

    if not li:
        print(f"Tidak ada data yang dibaca dari file CSV untuk prefix: {keyword_prefix}_x_reviews")
        return pd.DataFrame()

    frame = pd.concat(li, axis=0, ignore_index=True)
    original_count = len(frame)
    # Deduplicate based on 'text' column, keeping the first occurrence
    frame.drop_duplicates(subset=['text'], keep='first', inplace=True)
    deduplicated_count = len(frame)
    print(f"Menggabungkan {len(all_files)} file untuk {keyword_prefix}. Total {original_count} baris, setelah deduplikasi {deduplicated_count} baris.")
    return frame


# Proses untuk ClickUp
clickup_merged_df = merge_and_deduplicate_csv('clickup')
if not clickup_merged_df.empty:
    clickup_merged_df.to_csv('merged_clickup_x_reviews.csv', index=False)
    print("Preview data gabungan ClickUp:")
    display(clickup_merged_df.head())
    print(f"Total baris di merged_clickup_x_reviews.csv: {len(clickup_merged_df)}")


# Proses untuk Jira
jira_merged_df = merge_and_deduplicate_csv('jira')
if not jira_merged_df.empty:
    jira_merged_df.to_csv('merged_jira_x_reviews.csv', index=False)
    print("\nPreview data gabungan Jira:")
    display(jira_merged_df.head())
    print(f"Total baris di merged_jira_x_reviews.csv: {len(jira_merged_df)}")


# Proses untuk Trello
trello_merged_df = merge_and_deduplicate_csv('trello')
if not trello_merged_df.empty:
    trello_merged_df.to_csv('merged_trello_x_reviews.csv', index=False)
    print("\nPreview data gabungan Trello:")
    display(trello_merged_df.head())
    print(f"Total baris di merged_trello_x_reviews.csv: {len(trello_merged_df)}")


Menggabungkan 3 file untuk clickup. Total 233 baris, setelah deduplikasi 166 baris.
Preview data gabungan ClickUp:


,timestamp,text,target_keyword
0,2026-04-17T13:57:24.000Z,I just saw ClickUp launch AI agents that work ...,@clickup OR clickup OR #clickup
1,2026-04-23T13:00:37.000Z,AI Tools Ecosystem ┃ ┣ Time Management ┃ ┣ A...,@clickup OR clickup OR #clickup
2,2026-04-10T09:20:00.000Z,The biggest problem with clickup is that it al...,@clickup OR clickup OR #clickup
3,2026-04-15T16:39:13.000Z,I built a custom agent inside my @clickup wo...,@clickup OR clickup OR #clickup
4,2026-03-26T16:24:25.000Z,Top AI Productivity Tools You MUST Try in 2026...,@clickup OR clickup OR #clickup


Total baris di merged_clickup_x_reviews.csv: 166
Menggabungkan 3 file untuk jira. Total 334 baris, setelah deduplikasi 249 baris.

Preview data gabungan Jira:


,timestamp,text,target_keyword
0,2026-04-25T08:00:32.000Z,DEVOPS LAYERS PLANNING LAYER → Defines projec...,@jira OR jira OR #jira
1,2026-04-22T11:40:22.000Z,Your company's knowledge is trapped. It's sca...,@jira OR jira OR #jira
2,2026-04-25T11:58:41.000Z,( ◜ϖ◝ ).;:…( ◜ϖ◝ ...:.;::..( ◜;::: .:.;:...,@jira OR jira OR #jira
3,2026-04-20T14:32:04.000Z,New policy from @Atlassian : Unless you opt...,@jira OR jira OR #jira
4,2026-04-25T12:33:11.000Z,NaN,@jira OR jira OR #jira


Total baris di merged_jira_x_reviews.csv: 249
Menggabungkan 3 file untuk trello. Total 401 baris, setelah deduplikasi 285 baris.

Preview data gabungan Trello:


,timestamp,text,target_keyword
0,2026-04-23T18:23:51.000Z,A little over 1 week since officially launched...,@trello
1,2026-04-23T11:52:00.000Z,You could also consider http:// ProofHub.com ...,@trello
2,2026-04-22T18:06:14.000Z,First open source project collboration of our ...,@trello
3,2026-04-22T18:00:49.000Z,"The wait is finally over. Project ""Noobieteam""...",@trello
4,2026-04-22T06:32:39.000Z,Strategy isn't a ladder; it's an ecosystem. ...,@trello


Total baris di merged_trello_x_reviews.csv: 285


In [24]:
clickup_merged_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 166 entries, 0 to 181
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   timestamp       166 non-null    object
 1   text            165 non-null    object
 2   target_keyword  166 non-null    object
dtypes: object(3)
memory usage: 5.2+ KB


In [29]:
clickup_merged_df.duplicated().sum()

np.int64(0)

In [25]:
jira_merged_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 249 entries, 0 to 333
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   timestamp       249 non-null    object
 1   text            248 non-null    object
 2   target_keyword  249 non-null    object
dtypes: object(3)
memory usage: 7.8+ KB


In [28]:
jira_merged_df.duplicated().sum()

np.int64(0)

In [26]:
trello_merged_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 285 entries, 0 to 400
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   timestamp       285 non-null    object
 1   text            284 non-null    object
 2   target_keyword  285 non-null    object
dtypes: object(3)
memory usage: 8.9+ KB


In [27]:
trello_merged_df.duplicated().sum()

np.int64(0)